In [1]:
"""
Build basis_master sheet: fc_demographics
-----
Family Cover demographic assumptions by age: pct_married, n_children_0_13, n_children_14_plus.
Source: Basis and Assumptions.xlsx > FC Basis - New > G3:J103.

Header row: ANB | MARRIAGE_PERC | AGED_0_TO_13 | AGED_14_PLUS
Age range: 16..115 (100 rows).

Normalization: MARRIAGE_PERC source stores 86.0 meaning 86%. Normalised to decimal here.

Author: Mohamed Hussein Gadiwala
Basis version: v2026_07 (basis master)
"""

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import csv
import openpyxl
import polars as pl

BASE = Path(r"J:\Product & Client Solutions\Toolbox\Calculators"
            r"\Business Tools\Tender Summary\GAP Pricing Engine")

SOURCE_XLSX  = BASE / "Basis and Assumptions.xlsx"
SOURCE_SHEET = "FC Basis - New"

BASIS_VERSION = "v2026_07"
SNAPSHOT_DIR = BASE / "basis_snapshots" / f"basis_{BASIS_VERSION}"
MASTER_XLSX  = BASE / "basis_master" / f"basis_master_{BASIS_VERSION}.xlsx"
BUILD_LOG    = SNAPSHOT_DIR / "build_log.csv"

SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)
MASTER_XLSX.parent.mkdir(parents=True, exist_ok=True)

SHEET_NAME   = "fc_demographics"
PARQUET_PATH = SNAPSHOT_DIR / f"{SHEET_NAME}.parquet"

BUILT_AT = datetime.now(timezone.utc).isoformat(timespec="seconds")
BUILT_BY = "Mohamed Hussein Gadiwala"

wb = openpyxl.load_workbook(SOURCE_XLSX, data_only=True)
ws = wb[SOURCE_SHEET]
SOURCE_SHA = hashlib.sha256(SOURCE_XLSX.read_bytes()).hexdigest()

print(f"Building: {SHEET_NAME}")
print(f"Source:   {SOURCE_XLSX.name} > {SOURCE_SHEET}")
print(f"Target:   {PARQUET_PATH}")
print(f"Built at: {BUILT_AT}")


Building: fc_demographics
Source:   Basis and Assumptions.xlsx > FC Basis - New
Target:   J:\Product & Client Solutions\Toolbox\Calculators\Business Tools\Tender Summary\GAP Pricing Engine\basis_snapshots\basis_v2026_07\fc_demographics.parquet
Built at: 2026-07-22T10:47:08+00:00


In [2]:
def find_all_cells_any_col(ws, label: str, max_row: int = 1200, max_col: int = 30) -> list[tuple[int, int]]:
    """Find ALL cells matching `label` exactly, anywhere on the sheet."""
    hits = []
    for row in range(1, max_row + 1):
        for col in range(1, max_col + 1):
            v = ws.cell(row=row, column=col).value
            if isinstance(v, str) and v.strip() == label:
                hits.append((row, col))
    return hits


def find_in_range(ws, label: str, row_start: int, row_end: int, col: int | None = None) -> tuple[int, int] | None:
    """Find first cell matching `label` within given row range (and optional col)."""
    hits = find_all_cells_any_col(ws, label)
    for r, c in hits:
        if row_start <= r < row_end and (col is None or c == col):
            return (r, c)
    return None


def get_col_letter(col: int) -> str:
    return openpyxl.utils.get_column_letter(col)


In [3]:
expected_headers = ["ANB", "MARRIAGE_PERC", "AGED_0_TO_13", "AGED_14_PLUS"]
actual_headers = [ws.cell(row=3, column=c).value for c in (7, 8, 9, 10)]
assert actual_headers == expected_headers, (
    f"Header drift: got {actual_headers}, expected {expected_headers}"
)

DATA_ROW_START = 4
DATA_ROW_END   = 104
EXPECTED_AGES  = list(range(16, 116))

all_rows: list[dict] = []
for excel_row in range(DATA_ROW_START, DATA_ROW_END):
    age_raw          = ws.cell(row=excel_row, column=7).value
    marriage_raw     = ws.cell(row=excel_row, column=8).value
    kids_0_13_raw    = ws.cell(row=excel_row, column=9).value
    kids_14_plus_raw = ws.cell(row=excel_row, column=10).value

    if age_raw is None:
        raise RuntimeError(f"Null age at row {excel_row}")

    all_rows.append({
        "product":            "fc",
        "age":                int(age_raw),
        "pct_married":        float(marriage_raw) / 100.0 if marriage_raw is not None else 0.0,
        "n_children_0_13":    float(kids_0_13_raw)    if kids_0_13_raw    is not None else 0.0,
        "n_children_14_plus": float(kids_14_plus_raw) if kids_14_plus_raw is not None else 0.0,
    })

print(f"Total rows harvested: {len(all_rows)}")
print("First 3:", all_rows[:3])
print("Last 3:",  all_rows[-3:])


Total rows harvested: 100
First 3: [{'product': 'fc', 'age': 16, 'pct_married': 0.0, 'n_children_0_13': 1.0, 'n_children_14_plus': 0.0}, {'product': 'fc', 'age': 17, 'pct_married': 0.0, 'n_children_0_13': 1.0, 'n_children_14_plus': 0.0}, {'product': 'fc', 'age': 18, 'pct_married': 0.01, 'n_children_0_13': 1.0, 'n_children_14_plus': 0.0}]
Last 3: [{'product': 'fc', 'age': 113, 'pct_married': 0.49, 'n_children_0_13': 0.0, 'n_children_14_plus': 0.0}, {'product': 'fc', 'age': 114, 'pct_married': 0.49, 'n_children_0_13': 0.0, 'n_children_14_plus': 0.0}, {'product': 'fc', 'age': 115, 'pct_married': 0.49, 'n_children_0_13': 0.0, 'n_children_14_plus': 0.0}]


In [4]:
df = pl.DataFrame(all_rows).select([
    "product", "age", "pct_married", "n_children_0_13", "n_children_14_plus",
])

dupes = df.group_by(["product", "age"]).len().filter(pl.col("len") > 1)
assert dupes.height == 0

assert set(df["product"].unique().to_list()) == {"fc"}

ages = sorted(df["age"].to_list())
assert ages == EXPECTED_AGES, f"Age drift: {ages[0]}..{ages[-1]}"

assert df["pct_married"].max() <= 1.0
assert df["pct_married"].min() >= 0.0
assert df["n_children_0_13"].min() >= 0.0
assert df["n_children_14_plus"].min() >= 0.0

def get_row(age):
    row = df.filter(pl.col("age") == age)
    return row.row(0, named=True)

r = get_row(16)
assert r["pct_married"] == 0.0 and r["n_children_0_13"] == 1.0

r = get_row(40)
assert abs(r["pct_married"] - 0.86) < 1e-9
assert r["n_children_0_13"] == 3.0 and r["n_children_14_plus"] == 1.0

r = get_row(65)
assert abs(r["pct_married"] - 0.85) < 1e-9
assert r["n_children_0_13"] == 0.0

r = get_row(115)
assert abs(r["pct_married"] - 0.49) < 1e-9

print(f"✓ {df.height} rows validated")
print(f"✓ ages contiguous 16..115")
print(f"✓ pct_married normalised (max={df['pct_married'].max():.2f})")
print(f"✓ frozen checks passed")
print(df.head(3))
print(df.tail(3))


✓ 100 rows validated
✓ ages contiguous 16..115
✓ pct_married normalised (max=0.88)
✓ frozen checks passed
shape: (3, 5)
┌─────────┬─────┬─────────────┬─────────────────┬────────────────────┐
│ product ┆ age ┆ pct_married ┆ n_children_0_13 ┆ n_children_14_plus │
│ ---     ┆ --- ┆ ---         ┆ ---             ┆ ---                │
│ str     ┆ i64 ┆ f64         ┆ f64             ┆ f64                │
╞═════════╪═════╪═════════════╪═════════════════╪════════════════════╡
│ fc      ┆ 16  ┆ 0.0         ┆ 1.0             ┆ 0.0                │
│ fc      ┆ 17  ┆ 0.0         ┆ 1.0             ┆ 0.0                │
│ fc      ┆ 18  ┆ 0.01        ┆ 1.0             ┆ 0.0                │
└─────────┴─────┴─────────────┴─────────────────┴────────────────────┘
shape: (3, 5)
┌─────────┬─────┬─────────────┬─────────────────┬────────────────────┐
│ product ┆ age ┆ pct_married ┆ n_children_0_13 ┆ n_children_14_plus │
│ ---     ┆ --- ┆ ---         ┆ ---             ┆ ---                │
│ str     ┆ i6

In [5]:
df.write_parquet(PARQUET_PATH)
parquet_bytes = PARQUET_PATH.read_bytes()
PARQUET_SHA = hashlib.sha256(parquet_bytes).hexdigest()
PARQUET_SIZE_KB = len(parquet_bytes) / 1024

print(f"✓ Wrote {PARQUET_PATH.name} ({PARQUET_SIZE_KB:.2f} KB)")

if MASTER_XLSX.exists():
    master_wb = openpyxl.load_workbook(MASTER_XLSX)
else:
    master_wb = openpyxl.Workbook()
    if "Sheet" in master_wb.sheetnames:
        del master_wb["Sheet"]

if SHEET_NAME in master_wb.sheetnames:
    del master_wb[SHEET_NAME]

ws_out = master_wb.create_sheet(SHEET_NAME)
ws_out.append(df.columns)
for row in df.iter_rows():
    ws_out.append(list(row))

master_wb.save(MASTER_XLSX)
print(f"✓ Master workbook updated")


✓ Wrote fc_demographics.parquet (2.27 KB)
✓ Master workbook updated


In [6]:
LOG_HEADERS = [
    "built_at", "built_by", "sheet", "rows", "cols",
    "parquet_path", "parquet_size_kb", "parquet_sha256",
    "source_file", "source_sheet", "source_sha256",
    "basis_version", "notes",
]

log_row = {
    "built_at": BUILT_AT, "built_by": BUILT_BY, "sheet": SHEET_NAME,
    "rows": df.height, "cols": len(df.columns),
    "parquet_path": str(PARQUET_PATH.relative_to(BASE)),
    "parquet_size_kb": f"{PARQUET_SIZE_KB:.2f}", "parquet_sha256": PARQUET_SHA,
    "source_file": SOURCE_XLSX.name, "source_sheet": SOURCE_SHEET,
    "source_sha256": SOURCE_SHA, "basis_version": BASIS_VERSION,
    "notes": ("FC demographics by age (16..115, 100 rows). pct_married "
              "normalised to decimal at build time. Consumer: FC row-4 AB/AC/AD."),
}

write_header = not BUILD_LOG.exists()
with BUILD_LOG.open("a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=LOG_HEADERS)
    if write_header:
        writer.writeheader()
    writer.writerow(log_row)

print(f"✓ build_log.csv updated")
print(f"\n{'='*60}\nBUILD COMPLETE: {SHEET_NAME}\n{'='*60}")


✓ build_log.csv updated

BUILD COMPLETE: fc_demographics
